# 15 — Speech + Document Workflows — Voice-Driven Knowledge Access

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/15_speech_plus_document_workflows.ipynb)

> **Reading time:** ~30 min · **Prerequisites:** Notebooks 1–14 (especially **Notebook 13 — Speech Recognition & Transcription**)

This notebook connects two modalities that rarely appear in the same codebase but
belong together in every real product: **spoken language** and **written documents**.
We build end-to-end pipelines that let a user *speak* a question and receive an
answer grounded in a document corpus — no keyboard required.

## Learning Objectives

By the end of this notebook you will be able to:

1. **Wire Whisper ASR into a document-retrieval pipeline** so a spoken query triggers nearest-neighbour search over embedded passages.
2. **Build a FAISS index** from sentence-transformer embeddings and retrieve passages in under 10 ms.
3. **Generate grounded answers** using the Retrieval-Augmented Generation (RAG) pattern with Qwen3-8B in 4-bit quantization.
4. **Process meeting transcripts** — summarise, extract action items, and output structured JSON.
5. **Quantify error propagation** — measure how ASR word-error-rate degrades downstream retrieval quality.
6. **Apply mitigation strategies** — query expansion, multi-hypothesis retrieval, and confidence-based fallbacks.

## The Voice-to-Knowledge Use Case

Keyboards assume a desk. Screens assume your eyes are free. Neither assumption
holds for a surgeon mid-operation, a field technician crouched inside a turbine
housing, or a truck driver navigating a highway. **Voice is the universal
hands-free interface**, and when paired with a knowledge base it becomes
something far more powerful than a simple assistant — it becomes an expert on
call.

The canonical pipeline runs in five stages:

1. **Capture** — a microphone records the user’s spoken question (typically 5–15 s).
2. **Transcribe** — an ASR model (Whisper) converts the audio waveform into text.
3. **Search** — the transcribed query is embedded and matched against a document index.
4. **Generate** — an LLM synthesises a natural-language answer from the retrieved passages.
5. **Speak** (optional) — a TTS model converts the answer back to audio.

### Where This Is Already Deployed

| Domain | Example |
|---|---|
| **Enterprise assistants** | Field engineers dictate a fault description; the system retrieves the relevant repair procedure from a 10 000-page manual. |
| **Medical dictation + lookup** | A clinician describes symptoms aloud; the system matches against clinical guidelines and returns dosing recommendations. |
| **Accessibility** | Visually impaired users query government-form databases by voice and receive spoken step-by-step instructions. |

Every stage introduces latency and every stage can introduce errors.
This notebook teaches you to **build the pipeline** and, equally important,
to **measure and mitigate its failure modes**.

## Pipeline Architecture

```
┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐
│  Mic Audio │──▶│  Whisper   │──▶│  Embedding │──▶│   FAISS    │──▶│  Qwen LLM  │
│ (waveform) │   │   (ASR)   │   │  (S-BERT)  │   │ (retrieval)│   │ (RAG gen) │
└────────────┘   └────────────┘   └────────────┘   └────────────┘   └────────────┘
   ~0 ms          ~300 ms        ~20 ms          ~5 ms         ~2 000 ms
```

### Latency Budget

For conversational feel the round-trip must stay under **3 seconds**. The
bottleneck is almost always the LLM generation step. On a T4 GPU with
4-bit quantization, Qwen3-8B generates roughly 30 tokens/s, so a 60-token
answer takes ~2 s — leaving just 1 s for everything else.

### Modular vs End-to-End

A modular design (separate ASR → retriever → generator) is **easier to debug**
because you can inspect each intermediate output. An end-to-end model (e.g.
AudioPaLM) may achieve lower latency but is a black box. Throughout this
notebook we choose the modular path — it matches the skill level we have
built across Notebooks 1–14 and gives us more places to intervene when things
go wrong.

In [ ]:
# ── Installs (Colab) ───────────────────────────────────────────────
!pip install -q transformers torch torchaudio sentence-transformers \
    faiss-cpu datasets librosa bitsandbytes accelerate

In [ ]:
import json, textwrap, random, re, time
from pathlib import Path

import numpy as np
import torch
import torchaudio
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Building a Document Corpus

We need a corpus to search over. Below we create a **synthetic operations
manual** for the fictional *Ares-7 Mars Colony*. Each paragraph covers a
different topic — life support, power, agriculture, EVA procedures, etc.
This gives us enough topical diversity to test whether retrieval actually
returns the right passage for a given voice query.

In [ ]:
CORPUS = [
    "The Ares-7 life-support module recycles atmospheric gases using a "
    "closed-loop photocatalytic system. Oxygen is generated from CO2 by "
    "titanium-dioxide reactors illuminated with UV-C LEDs running at 265 nm. "
    "Backup oxygen is stored in cryogenic tanks rated for 30 days of colony demand.",

    "Power for the colony is supplied by a hybrid system comprising 4.2 MW of "
    "thin-film perovskite solar panels and two 500 kW radioisotope thermoelectric "
    "generators (RTGs). During dust storms, which can reduce solar output by 90%, "
    "the RTGs alone sustain critical systems for up to 14 days.",

    "The agricultural dome uses aeroponics trays arranged in rotating vertical "
    "columns. Each column completes one revolution every 47 minutes to equalise "
    "light exposure. Primary crops include dwarf wheat, soybeans, potatoes, and "
    "kale. Nutrient solution pH is maintained at 5.8-6.2 by automated dosing.",

    "Extra-vehicular activity (EVA) suits are pressurised to 29.6 kPa with a "
    "nitrogen-oxygen mix (60/40). Suit battery packs provide eight hours of "
    "thermal regulation and CO2 scrubbing. All EVAs require buddy pairing and "
    "real-time telemetry to Mission Control.",

    "Water extraction relies on subsurface ice mining at the Noctis Labyrinthus "
    "deposit. A rotary drill with heated auger melts ice in situ; melt-water is "
    "pumped to the colony via insulated conduit and purified with UV and reverse-"
    "osmosis membranes. Daily extraction target is 800 litres.",

    "Communications with Earth use X-band uplink at 8.4 GHz through a 3.7-metre "
    "high-gain dish. Light-speed delay ranges from 4 to 24 minutes depending on "
    "orbital positions. Emergency messages use a redundant UHF relay via orbiting "
    "satellites.",

    "Medical bay is equipped for emergency surgery, dental procedures, and "
    "diagnostic imaging (portable CT). Pharmaceutical inventory covers antibiotics, "
    "analgesics, anaesthetics, and antiarrhythmics. Telemedicine consultations with "
    "Earth specialists are scheduled twice weekly.",

    "Radiation shielding consists of 2.5 metres of regolith packed above habitation "
    "modules. During solar particle events colonists shelter in the central storm "
    "cellar, which adds an additional 50 g/cm² of polyethylene shielding. Annual "
    "dose target is below 250 mSv.",

    "Waste management converts organic waste into methane and compost via anaerobic "
    "digestion. The methane feeds a fuel cell that supplements colony power. Non-"
    "organic waste is compacted, catalogued, and stored for eventual Earth-return "
    "cargo missions.",

    "Rover fleet comprises four pressurised exploration vehicles with a range of "
    "200 km each. Navigation uses a combination of inertial measurement units, "
    "star-tracker cameras, and pre-loaded HiRISE terrain maps. Maximum safe speed "
    "on prepared tracks is 35 km/h."
]

for i, para in enumerate(CORPUS):
    print(f"[Doc {i}] {para[:90]}...")
print(f"\nCorpus size: {len(CORPUS)} passages")

## Create a Document Index with FAISS

We embed every passage with `all-MiniLM-L6-v2` (384-d vectors, fast enough
for real-time use) and build a flat inner-product FAISS index. For our
10-document corpus a flat index is fine; in production you would switch to
`IndexIVFFlat` or `IndexHNSW` for sub-linear search.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

corpus_embeddings = embedder.encode(CORPUS, convert_to_numpy=True,
                                    normalize_embeddings=True)
dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product on L2-normalised vecs = cosine
index.add(corpus_embeddings)
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

In [ ]:
def retrieve(query: str, k: int = 3):
    """Embed a text query and return top-k passages with scores."""
    q_emb = embedder.encode([query], convert_to_numpy=True,
                            normalize_embeddings=True)
    scores, ids = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append({"score": float(score), "doc_id": int(idx),
                        "text": CORPUS[idx]})
    return results

for hit in retrieve("How does the colony generate electricity?"):
    print(f"  [{hit['doc_id']}] score={hit['score']:.3f}  {hit['text'][:80]}...")

## Step 1 — Transcribe Voice Queries

A voice query is typically short — 5 to 15 seconds — but accuracy requirements
are **higher** than for long-form transcription. A single misrecognised word in
a 6-word question can flip the meaning entirely: *"What is the water
**extraction** rate?"* vs *"What is the water **exaction** rate?"*

We use **Whisper-small** (`openai/whisper-small`) — 244 M parameters, roughly
3× real-time on a T4 GPU. For queries under 10 s that translates to
under 300 ms wall-clock. Whisper also returns token-level log-probabilities
that we will exploit later for confidence scoring.

In [ ]:
whisper_proc = AutoProcessor.from_pretrained("openai/whisper-small")
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-small", torch_dtype=torch.float16
).to(DEVICE)

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_proc.tokenizer,
    feature_extractor=whisper_proc.feature_extractor,
    torch_dtype=torch.float16,
    device=DEVICE,
)
print("Whisper-small loaded.")

In [ ]:
# Load a short speech sample from HuggingFace datasets
speech_ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean",
    split="validation", trust_remote_code=True,
)
sample = speech_ds[0]
audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]

result = asr_pipe(
    {"raw": audio_array, "sampling_rate": sr},
    return_timestamps=True,
)
transcription = result["text"].strip()
print(f"Transcription: {transcription}")
print(f"Audio duration: {len(audio_array)/sr:.1f}s")

In [ ]:
# For the RAG demo we simulate a voice query that matches our corpus.
# In production this text would come from Whisper.
voice_query = "How does the colony get its water supply?"
print(f"Simulated voice query: {voice_query}")

## Step 2 — Retrieve Relevant Documents

The transcribed text becomes a query vector via the same sentence-transformer
used to index the corpus. We perform a **nearest-neighbour search** in FAISS
and return the top-*k* passages (typically *k* = 3).

Relevance scoring is cosine similarity (because we L2-normalised both
corpus and query embeddings before indexing). Scores above 0.5 usually
indicate strong topical overlap; below 0.3 is noise. Choosing *k* is a
precision–recall trade-off: more passages give the LLM more material but
also more noise and a longer prompt.

In [ ]:
hits = retrieve(voice_query, k=3)
print(f'Query: "{voice_query}"\n')
for rank, hit in enumerate(hits, 1):
    print(f"  Rank {rank} [doc {hit['doc_id']}] score={hit['score']:.3f}")
    print(f"    {hit['text'][:120]}...\n")

## Step 3 — Generate an Answer with an LLM

We follow the **Retrieval-Augmented Generation (RAG)** pattern introduced by
Lewis et al. (2020): retrieved passages are *stuffed* into the prompt as
context, and the LLM is instructed to answer *only* from that context.

We use **Qwen/Qwen3-8B** with 4-bit NF4 quantization via `bitsandbytes`.
This reduces VRAM from ~16 GB to ~5 GB — comfortably within T4 limits.
The prompt template is deliberately simple: a system instruction, the
context block, and the question. Fancy prompt engineering is left for
production; here we focus on the plumbing.

In [ ]:
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_cfg,
    device_map="auto",
)
print(f"Qwen3-8B loaded in 4-bit on {llm.device}")

In [ ]:
def rag_answer(question: str, context_hits: list, max_new_tokens: int = 200):
    """Generate a RAG answer from retrieved passages."""
    context_block = "\n\n".join(
        f"[Passage {i+1}] {h['text']}" for i, h in enumerate(context_hits)
    )
    messages = [
        {"role": "system", "content": (
            "You are an operations assistant for the Ares-7 Mars Colony. "
            "Answer the question using ONLY the provided context. "
            "If the context does not contain the answer, say so."
        )},
        {"role": "user", "content": (
            f"Context:\n{context_block}\n\nQuestion: {question}"
        )},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                           do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)

answer = rag_answer(voice_query, hits)
print(f"Q: {voice_query}")
print(f"A: {answer}")

In [ ]:
def voice_to_answer(query_text: str, k: int = 3):
    """End-to-end: text query → retrieval → RAG answer."""
    t0 = time.time()
    hits = retrieve(query_text, k=k)
    t_retrieve = time.time() - t0
    t1 = time.time()
    ans = rag_answer(query_text, hits)
    t_gen = time.time() - t1
    return {
        "query": query_text,
        "answer": ans,
        "retrieve_ms": round(t_retrieve * 1000, 1),
        "generate_ms": round(t_gen * 1000, 1),
        "hits": hits,
    }

r = voice_to_answer("What crops are grown in the agricultural dome?")
print(f"Q: {r['query']}")
print(f"A: {r['answer']}")
print(f"⏱ Retrieval: {r['retrieve_ms']} ms  |  Generation: {r['generate_ms']} ms")

## Meeting Note Processing

Voice pipelines are not limited to short queries. A common enterprise
workflow is: **record a meeting → transcribe → extract structured data**.

Meeting audio introduces challenges absent in short queries:

* **Length** — meetings run 30–60 minutes; Whisper chunks at 30 s boundaries.
* **Multiple speakers** — turn-taking, cross-talk, and interruptions.
* **Domain jargon** — project names, acronyms, and internal lingo.

Below we simulate a meeting transcript and use the LLM to extract
summaries, action items, and key decisions in JSON format.

In [ ]:
MEETING_TRANSCRIPT = """
Alice: Good morning everyone. Let's start with the water extraction update.
Bob: We hit 780 litres yesterday, slightly below the 800 target. The auger
drill bit is showing wear. I've requested a replacement from inventory.
Alice: How long until the replacement is installed?
Bob: Two days if we pull it from storage bay C. I'll handle it tomorrow.
Carol: On the ag side, the kale trays had a pH spike to 6.8 overnight. I
adjusted the dosing pump and it's back to 6.1 now.
Alice: Good catch, Carol. Should we add a pH alarm threshold?
Carol: Yes, I'll configure an alert at 6.5 by end of week.
Dave: Comms update — the next Earth window opens in 47 minutes. I have the
weekly telemetry package ready to upload.
Alice: Perfect. Let's wrap up. Action items: Bob replaces drill bit, Carol
sets pH alarm, Dave uploads telemetry. Next meeting same time Thursday.
""".strip()

extract_prompt = [
    {"role": "system", "content": (
        "You are a meeting assistant. Extract structured data from the "
        "transcript and return ONLY valid JSON with keys: summary, "
        "action_items (list of {owner, task, deadline}), decisions "
        "(list of strings), participants (list of strings)."
    )},
    {"role": "user", "content": f"Transcript:\n{MEETING_TRANSCRIPT}"},
]

prompt_text = tokenizer.apply_chat_template(
    extract_prompt, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt_text, return_tensors="pt").to(llm.device)
with torch.no_grad():
    out = llm.generate(**inputs, max_new_tokens=400, do_sample=False)
meeting_output = tokenizer.decode(
    out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print(meeting_output)

## Error Propagation in Voice Pipelines

ASR is imperfect. Whisper-small achieves roughly 4–5 % word error rate
(WER) on clean English speech, but WER on domain-specific or noisy audio
can climb to 15 %+. The critical insight is:

> **10 % WER does not mean 10 % answer degradation — it can be much worse.**

A single substitution on a **content word** (noun, verb, number) can
completely change the query meaning and cause the retriever to return
irrelevant passages. Common failure patterns:

| Failure Type | Example | Impact |
|---|---|---|
| Named entity | "Noctis" → "Notice" | Retrieves wrong passage |
| Technical term | "perovskite" → "prevalent" | Total mismatch |
| Number | "29.6 kPa" → "26 kPa" | Wrong specification |
| Function word | "not pressurised" → "pressurised" | Semantic inversion |

Below we quantify this with a controlled experiment.

In [ ]:
def corrupt_query(query: str, n_substitutions: int, seed: int = 42):
    """Replace n random content words with plausible but wrong words."""
    rng = random.Random(seed)
    noise_words = ["system", "module", "process", "general",
                   "standard", "normal", "basic", "common"]
    words = query.split()
    # Only substitute words with length > 3 (skip articles, etc.)
    eligible = [i for i, w in enumerate(words) if len(w) > 3]
    to_replace = rng.sample(eligible, min(n_substitutions, len(eligible)))
    for i in to_replace:
        words[i] = rng.choice(noise_words)
    return " ".join(words)

ref_queries = [
    ("How does the colony generate electricity from solar panels?", 1),
    ("What is the daily water extraction target in litres?", 4),
    ("Describe the radiation shielding above habitation modules.", 7),
]

print("Retrieval Recall@3 — clean vs corrupted queries\n")
for query, expected_doc in ref_queries:
    clean_hits = {h["doc_id"] for h in retrieve(query, k=3)}
    results = {"clean": int(expected_doc in clean_hits)}
    for n_err in [1, 2, 3]:
        corrupted = corrupt_query(query, n_err)
        noisy_hits = {h["doc_id"] for h in retrieve(corrupted, k=3)}
        results[f"{n_err}_err"] = int(expected_doc in noisy_hits)
    print(f'  Q: "{query[:50]}..."')
    print(f"    Recall@3 -> clean={results['clean']}  "
          f"1-err={results['1_err']}  "
          f"2-err={results['2_err']}  "
          f"3-err={results['3_err']}\n")

## Mitigation Strategies

When ASR errors are inevitable, the pipeline should be **resilient** rather
than brittle. Practical mitigations include:

1. **N-best hypotheses** — Whisper (and most ASR models) can return the top-N
   transcription candidates. Retrieve for each and merge results.
2. **Query expansion** — append synonyms or related terms to broaden the
   retrieval net. Even a noisy expansion can recover a missed passage.
3. **Domain spell-correction** — a small dictionary of colony-specific terms
   (e.g. "Noctis", "perovskite", "aeroponics") catches phonetically plausible
   ASR errors.
4. **Confidence gating** — if the ASR confidence (mean log-prob) falls below
   a threshold, ask the user to repeat rather than proceeding with a bad
   transcript.
5. **Retrieval-score thresholding** — if no passage scores above 0.3, return
   "I didn’t find a relevant answer" instead of hallucinating.

In [ ]:
def multi_hypothesis_retrieve(hypotheses: list, k: int = 3):
    """Retrieve for multiple ASR hypotheses and merge results."""
    seen = {}
    for hyp in hypotheses:
        for hit in retrieve(hyp, k=k):
            doc_id = hit["doc_id"]
            if doc_id not in seen or hit["score"] > seen[doc_id]["score"]:
                seen[doc_id] = hit
    merged = sorted(seen.values(), key=lambda x: x["score"], reverse=True)
    return merged[:k]

# Simulate N-best from ASR: correct + two noisy variants
hyp_1 = "What is the daily water extraction target in litres?"
hyp_2 = "What is the daily water module target in litres?"   # 1 error
hyp_3 = "What is the basic process standard in litres?"      # 3 errors

single = retrieve(hyp_2, k=3)  # using the 1-error hypothesis alone
merged = multi_hypothesis_retrieve([hyp_1, hyp_2, hyp_3], k=3)

print("Single-hypothesis retrieval (1-error):")
for h in single:
    print(f"  doc {h['doc_id']} score={h['score']:.3f}")

print("\nMulti-hypothesis retrieval (merged):")
for h in merged:
    print(f"  doc {h['doc_id']} score={h['score']:.3f}")

In [ ]:
DOMAIN_DICT = {
    "notice": "Noctis",
    "prevalent": "perovskite",
    "arrow ponics": "aeroponics",
    "error ponics": "aeroponics",
    "rtg": "RTG",
    "arise": "Ares",
}

def domain_correct(text: str) -> str:
    """Replace known ASR errors with domain-specific terms."""
    corrected = text.lower()
    for wrong, right in DOMAIN_DICT.items():
        corrected = corrected.replace(wrong, right)
    return corrected

noisy = "What prevalent panels does the arise colony use?"
fixed = domain_correct(noisy)
print(f"Before: {noisy}")
print(f"After:  {fixed}")
print()
for h in retrieve(fixed, k=2):
    print(f"  doc {h['doc_id']} score={h['score']:.3f}  {h['text'][:80]}...")

## Exercises

### Exercise 1 — Custom Voice-to-Knowledge Pipeline

Build a voice-to-answer pipeline for a **different** document corpus of your
choice (e.g. a product FAQ, a textbook chapter, or a set of cooking recipes).
Create at least 8 documents, build the FAISS index, and demonstrate 3 queries
that return relevant answers.

### Exercise 2 — WER vs Retrieval Recall

Write a function that takes a clean query, introduces word errors at rates of
5 %, 10 %, 15 %, 20 %, and 30 %, runs retrieval for each, and plots
Recall@3 as a function of WER. Run it on at least 5 different queries and
report the average degradation curve.

### Exercise 3 — Confidence-Based Fallback

Modify the ASR step to extract the **mean log-probability** of the
transcription. If the mean log-prob falls below a threshold (e.g. −0.5),
print a message asking the user to repeat their question instead of
proceeding with retrieval. Test with both clean and noisy audio.

In [ ]:
# ── Exercise 1 Starter ─────────────────────────────────────────────
MY_CORPUS = [
    # Add at least 8 documents here
    "Document 1 text...",
    "Document 2 text...",
]

# TODO: embed MY_CORPUS with the sentence-transformer
# TODO: build a new FAISS index
# TODO: write 3 queries and show retrieval + RAG answers

In [ ]:
# ── Exercise 2 Starter ─────────────────────────────────────────────
def measure_recall_vs_wer(query: str, expected_doc: int,
                          wer_levels=(0.05, 0.10, 0.15, 0.20, 0.30),
                          k: int = 3, n_trials: int = 10):
    """For each WER level, corrupt the query n_trials times and measure recall."""
    results = {}
    words = query.split()
    for wer in wer_levels:
        n_sub = max(1, int(len(words) * wer))
        hits_count = 0
        for trial in range(n_trials):
            noisy_q = corrupt_query(query, n_sub, seed=trial)
            hit_ids = {h["doc_id"] for h in retrieve(noisy_q, k=k)}
            hits_count += int(expected_doc in hit_ids)
        results[wer] = hits_count / n_trials
    return results

# TODO: run on 5+ queries, average the results, and plot with matplotlib

In [ ]:
# ── Exercise 3 Starter ─────────────────────────────────────────────
CONFIDENCE_THRESHOLD = -0.5

def transcribe_with_confidence(audio_arr, sampling_rate):
    """Transcribe and return (text, mean_log_prob)."""
    result = asr_pipe(
        {"raw": audio_arr, "sampling_rate": sampling_rate},
        return_timestamps=True,
    )
    text = result["text"].strip()
    # TODO: extract token-level log-probs from Whisper output
    # For now, use a placeholder confidence
    confidence = -0.3  # placeholder
    return text, confidence

def voice_pipeline_with_fallback(audio_arr, sr):
    text, conf = transcribe_with_confidence(audio_arr, sr)
    if conf < CONFIDENCE_THRESHOLD:
        return "Low confidence — please repeat your question."
    hits = retrieve(text, k=3)
    return rag_answer(text, hits)

# TODO: test with clean audio and with noisy audio

## Key Takeaways

| # | Insight |
|---|---|
| 1 | A voice-to-knowledge pipeline chains ASR → embedding → FAISS retrieval → LLM generation — each stage must be fast and inspectable. |
| 2 | Whisper-small transcribes short queries in under 300 ms on a T4 GPU, leaving the latency budget for generation. |
| 3 | FAISS with normalised sentence-transformer embeddings gives sub-10 ms cosine retrieval even at modest corpus sizes. |
| 4 | RAG with Qwen3-8B in 4-bit quantization generates grounded answers within T4 VRAM limits (~5 GB). |
| 5 | ASR errors propagate non-linearly: a 10 % WER can cause >30 % retrieval degradation on content-word substitutions. |
| 6 | Multi-hypothesis retrieval and domain spell-correction are low-cost mitigations that substantially improve robustness. |

## References

1. **Radford, A. et al.** (2023). *Robust Speech Recognition via Large-Scale Weak Supervision* (Whisper). [arXiv:2212.04356](https://arxiv.org/abs/2212.04356)
2. **Lewis, P. et al.** (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)
3. **Johnson, J., Douze, M., & Jégou, H.** (2021). *Billion-Scale Similarity Search with GPUs* (FAISS). [GitHub](https://github.com/facebookresearch/faiss)
4. **Reimers, N. & Gurevych, I.** (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
5. **Qwen Team** (2025). *Qwen3 Technical Report*. [Blog](https://qwenlm.github.io/blog/qwen3/)

---

← **[Notebook 14 — Audio Understanding, Classification & Tagging](14_audio_understanding_classification_and_tagging.ipynb)** ·
**[Notebook 16 — Multimodal Agents across Speech & Vision](16_multimodal_agents_across_speech_and_vision.ipynb)** →

*This notebook builds directly on the ASR foundations from
Notebook 13 and feeds into the multimodal agent orchestration
patterns explored in Notebook 16.*